In [53]:
# Import Required Libraries
import pandas as pd
import pickle

In [54]:
# Load Dataset
insurance_df = pd.read_csv("insurance_pre.csv")
insurance_df

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [55]:
# check for null values
insurance_df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
charges     0
dtype: int64

In [56]:
# One-Hot Encoding for categorical features
insurance_df = pd.get_dummies(insurance_df, dtype=int, drop_first=True)
insurance_df

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [58]:
# variables name
insurance_df.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [59]:
# Feature–Target split
X = insurance_df[['age', 'bmi', 'children',  'sex_male', 'smoker_yes']]
y = insurance_df['charges']

In [60]:
# Train–Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [61]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [66]:
# Model Training with GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

param_grid = {
    "kernel": ["linear", "poly", "rbf", "sigmoid"],
    "C": [10,100,1000,2000,3000],
    "gamma": ["auto", "scale"]
}

svr_grid_search = GridSearchCV(
    estimator = SVR(), 
    param_grid = param_grid,
    scoring = "r2",
    cv = 5,
    refit = True, 
    verbose = 3, 
    n_jobs = -1)

svr_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(cv=5, estimator=SVR(), n_jobs=-1,
             param_grid={'C': [10, 100, 1000, 2000, 3000],
                         'gamma': ['auto', 'scale'],
                         'kernel': ['linear', 'poly', 'rbf', 'sigmoid']},
             scoring='r2', verbose=3)

In [67]:
# Best parameters
print("Best Parameters:", svr_grid_search.best_params_)

Best Parameters: {'C': 3000, 'gamma': 'auto', 'kernel': 'rbf'}


In [77]:
# Model Prediction
y_pred = svr_grid_search.predict(X_test)

In [76]:
# # Model Evaluation
from sklearn.metrics import r2_score
r2_score_value = r2_score(y_test,y_pred )
print("R2 Score: ", r2_score_value)

R2 Score:  0.8464875870695387


In [78]:
# Save Model
MODEL_PATH = "best_svr_model.pkl"
pickle.dump(
    {"model": svr_grid_search.best_estimator_, "scaler": scaler},
    open(MODEL_PATH, 'wb')
)

In [79]:
# Load Model
loaded_data = pickle.load(open(MODEL_PATH, "rb"))
loaded_model = loaded_data["model"]
loaded_scaler= loaded_data["scaler"]

In [87]:
# User Inputs
age = int(input("Enter Age: "))
bmi = float(input("Enter BMI: "))
children = int(input("Enter number of children: "))
sex_male = int(input("Male (1) or Female (0): "))
smoker_yes = int(input("Smoker (1) or Non-smoker (0): "))


Enter Age:  28
Enter BMI:  20
Enter number of children:  0
Male (1) or Female (0):  1
Smoker (1) or Non-smoker (0):  0


In [88]:
# Validate User Input
def validate_user_input(age, bmi, children, sex_male, smoker_yes):
    if not (10 <= age <= 100):
        raise ValueError("Age must be between 1 and 100")

    if not (10 <= bmi <= 70):
        raise ValueError("BMI must be between 10 and 70")

    if not (0 <= children <= 10):
        raise ValueError("Children must be between 0 and 10")

    if sex_male not in [0,1]:
        raise ValueError("Sex must be 0 (Female) or 1 (Male)")

    if smoker_yes not in [0,1]:
        raise ValueError("Smoker must be 0 (No) or 1 (Yes)")
    

In [89]:
try:
    validate_user_input(age, bmi, children, sex_male, smoker_yes)
    
    # Preprocess Input
    user_input = [[age, bmi, children, sex_male, smoker_yes]]
    user_input_scaled = scaler.transform(user_input)

    # Prediction 
    predicted_charges = loaded_model.predict(user_input_scaled)
    print(f"Predicted Insurance Charges: {predicted_charges[0]:.2f}")

except ValueError as error:
    print("Input Error:", error)
    

Predicted Insurance Charges: 2785.78


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
